
<div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/it/thumb/e/e2/Stemma_unipi.svg/900px-Stemma_unipi.svg.png?20221226121859" alt="Alt text" title="Title text" style="width:300px;" />
</div>
<h1 align="center" style="height:60px;"> UNIVERSITY OF PISA</h1>
<h2 align="center" style="height:60px;"> MASTER’S DEGREE IN ARTIFICIAL INTELLIGENCE AND DATA ENGINEERING</h2>
<h3 align="center"> Multimedia Information Retrieval and Computer Vision</h3>
<div style="height: 40px;"></div>
<h2 align="center"> RAGING BULL Group</h2>
<div style="height: 80px;"></div>



<div style="display: flex; justify-content: space-around;">
    <div style="width:200px;">
        <p>Professor:</p>
        <ul>
            <li>Nicola Tonellotto</li>
        </ul>
    </div>
    <div style="width:200px;">
        <p>Students:</p>
        <ul>
            <li>Cleto Pellegrino</li>
            <li>Giuseppe Soriano</li>
            <li>Massimiliano Romani</li>
            <li>Francesco Boldrini</li>
        </ul>
    </div>
</div>
<div style="height: 80px;"></div>
<hr>
<h3 align="center">A.Y. 2023/2024</h3>
<hr>

<h1 align="center">Introduction</h1>

The scope of this project is to realise and evaluate a search engine, using what was teached during the lectures. The documentation is divided in subsection, one for each step of the realisation of the search engine. To ensure to have the maximum compatibility with every Notebook manager, in the following block of code there is the installation of all the packages that will be used in this project.

In [2]:
# Installation of all the packages that will be used in this project
%pip install joblib -q
%pip install time -q
# %pip install functools -q
%pip install unicodedata -q
%pip install wordninja -q
%pip install requests -q
%pip install tarfile -q
%pip install os -q
%pip install gc -q
%pip install re -q
%pip install string -q
%pip install nltk -q
%pip install heapq -q
%pip install math -q
%pip install collections -q
%pip install tqdm -q
%pip install matplotlib -q
%pip install scipy -q
%pip install polars -q
%pip install pandas -q
%pip install gdown -q
# It takes about 1 minute and 5 seconds if all the packages are installed

Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement unicodedata (from versions: none)
ERROR: No matching distribution found for unicodedata
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement tarfile (from versions: none)
ERROR: No matching distribution found for tarfile
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os
Note: you may need to restart the kernel

<h1 align="center">Dataset</h1>
The first step consists in downloading the dataset that will be used to build and evaluate the search engine. It is called <b>msmarco</b> and it contains:
<ul>
    <li>Documents</li>
    <li></li>
    <li></li>
    <li></li>
</ul>

The collection is first downloaded (if not present in the working directory) from <a href='https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz'>this link</a>, then it is unpacked from the '.gz' format into a '.tsv' format, usable in Python code.


In [3]:
import requests
import tarfile
import os
# url of the dataset
url = 'https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz'
# file name
local_filename = 'collection.tar.gz'
# this condition checks if the collection is already downloaded (present in the folder) or not. in the latter case it downloads it 
if not os.path.exists(local_filename):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                
extract_path_file = os.getcwd()+"/collection.tsv"

if not os.path.exists(extract_path_file):
    with tarfile.open(local_filename, 'r:gz') as tar:
        tar.extractall(path=os.getcwd())


This Garbage Collector helps cleaning the unnecessary variables that would otherwise occupy space in memory, causing longer computation times

In [4]:
import gc
gc.enable()

<h1 align="center">Preprocessing</h1>
<p>In this step, the dataset is preprocessed in order to obtain the final, clean form for every document, in order for each of them to be processed in the real computation of the document retrieval.</p> 
<p>Usual documents contains more informations than just words: for example, there are web page links, email addresses, which are not interesting in the context of retrieving documents but are still computed in the build index and document retrieval phases, therefore they must be eliminated to improve the performances. the preprocessing phase will be applied to future queries as well, otherwise the system would not be able to effectively return the documents.</p>
<p> The first step consists in defining the patterns for recognizing with regular expressions acronyms, links starting with "http", "www" and email addresses and creating the objects to delete all punctuation marks.
</p>

In [5]:
import re
import string
import nltk

from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

STEMMER, STOPWORDS = SnowballStemmer("english"), set(stopwords.words("english"))

# Acronyms management (U.S.A.) -> USA
# (Mr.Jhonsons) -> Mr.Jhonson
# If the fullstop is missing in the end, like in www.colab.com, the regex for acronyms (ACRONYM_RE) purposely doesn't work, because 
# in those cases there is the WWW_REGEX.
ACRONYM_RE = re.compile(r"\b(?:[A-Za-z]\.){2,}")

#Creates the object pattern that will be later used to find and substitute the string patterns in the documents resulting from the regular expression
HTTP_REGEX = re.compile(r"\s?http\S+")
WWW_REGEX = re.compile(r"\s?www\S+")
EMAIL_REGEX = re.compile(r"\S+@\S+")

CLEAN_TABLE = str.maketrans(string.punctuation, " " * len(string.punctuation))

<p>Here, the proper preprocess function is written: it takes as input a string, representing the content of a document and returns as output a list of strings representing the cleaned document. These are the steps:
<ol>
    <h4>
    <li>
        <b>Document text's is transformed into a comparable form by:</b></h4>
        <ol>
            <li><p><b>Normalising</b>: makes possible to determine if two Unicode strings (normalised in the same way) are equivalent. basically it changes composite symbols into their separate forms in the specific order established by the <b>Normalization form</b>, like in the example below: <br /> </p>
            <img src="https://unicode.org/reports/tr15/images/UAX15-NormFig6.jpg" style="width:450px;">
            <p>in this case, the <b>NFKD</b> (Normalisation Form: Compatibility Decomposition) option was chosen, which means that characters are decomposed, in order to obtain the base forms of the characters composing the original character, from the point of view of compatibility between characters. Two characters are compatible if they have the same significance even if the style form or the usage is different. This differs from Canonicity, in which two characters have the same significance from semantic and visual point of view, but the codes are different. </p> </li>
            <li><p><b>Encoding</b>: converts each Unicode character in a sequence of bytes using ASCII's set of characters, ignoring characters not representable in ASCII (thanks to the 'ignore' option enabled).</p></li>
            <li><p><b>Decoding</b>: converts back the sequence of bytes into a Unicode string that can be used as text in Python. The encode-decode step is used to eliminate all non-ASCII representable characters, to clean the documents.</p></li>
            <li><p><b>Lowering</b>: Converts all characters into their lowercase form.</p></li>
        </ol>
    </li>
    <h4><li>Acronyms are replaced by the same characters but without punctuation marks</li>
    <li>HTTP, WWW links and Email addresses are removed</li>
    <li>Punctuation marks are removed</li>
    <li>Additional spaces or tabulation characters are removed</li>
    <li>Single words are obtained by splitting strings on spaces</li>
    <li>Composite words, like "mywebsiteisawesome" of more than 10 words are split in single words</li>
    <li>Every word that is equal to the previous word is eliminated</li>
</ol>
</p>

In [6]:
from time import time
from functools import lru_cache
import unicodedata
import wordninja

@lru_cache(maxsize=None) # For preprocessing optimization
def stem_word(word):
    return STEMMER.stem(word)

@lru_cache(maxsize=None)
def split_word(word):
    return wordninja.split(word)

def preprocess(text):
    
    text = unicodedata.normalize("NFKD", text).encode('ASCII', 'ignore').decode("utf-8").lower()
    # .sub means "substitute" first element into second element, where the second element is a substring of the string representing the document
    # that verifies the regular expression
    text = ACRONYM_RE.sub(lambda x: x.group().replace('.', ''), text)

    text = HTTP_REGEX.sub(' ', text)
    text = WWW_REGEX.sub(' ', text)
    text = EMAIL_REGEX.sub(' ', text)

    # Discarding punctuation and other unuseful characters
    # It was purposely put at the end so that regexes have the opportunity to use punctuation marks like @ or fullstops for acronyms
    text = text.translate(CLEAN_TABLE)

    words = text.strip().split()
    
    words_with_no_close_duplicate = []
    current = ""
    for word in words:
        if word != current:
            splitted_words = split_word(word) if len(word) > 10 else [word]
            for splitted_word in splitted_words:
                if splitted_word not in STOPWORDS:
                    words_with_no_close_duplicate.append(stem_word(splitted_word))
            current = word
            
    return words_with_no_close_duplicate

<div style="background-color:orange;color:black;">
<h1 align="center"; style="color:red;">TEST</h1>
<p>Let's see if using NFKD has some difference with using NFKC, since the latter is used in search engines</p>

In [7]:
#THE TEST
def preprocess_NFKC(text):
    text = unicodedata.normalize("NFKC", text).encode('ASCII', 'ignore').decode("utf-8").lower()
    # .sub means "substitute" first element into second element, where the second element is a substring of the string representing the document
    # that verifies the regular expression
    text = ACRONYM_RE.sub(lambda x: x.group().replace('.', ''), text)

    text = HTTP_REGEX.sub(' ', text)
    text = WWW_REGEX.sub(' ', text)
    text = EMAIL_REGEX.sub(' ', text)

    # Discarding punctuation and other unuseful characters
    # It was purposely put at the end so that regexes have the opportunity to use punctuation marks like @ or fullstops for acronyms
    text = text.translate(CLEAN_TABLE)

    words = text.strip().split()
    
    words_with_no_close_duplicate = []
    current = ""
    for word in words:
        if word != current:
            splitted_words = split_word(word) if len(word) > 10 else [word]
            for splitted_word in splitted_words:
                if splitted_word not in STOPWORDS:
                    words_with_no_close_duplicate.append(stem_word(splitted_word))
            current = word
            
    return words_with_no_close_duplicate
queries = [
    "M4rk is a good programmer from U.S.A.!",
    "www.kaggle.com is a cool website, i love it!",
    "click this link for free 1000000 v-bucks, http://free-vbucks.com",
    "This is my new email, new_email@google.com",
    "Let`s go to this new Cafè, it's so good ♫ ☺!",
    "SPAM SPAM SPAM COPY COPY SPAM SPAM",
    "thissentenceisattachedbutwithwordninjawecansplitit"
]
for q in queries:
    NFKD_one = preprocess(q)
    NFKC_one = preprocess_NFKC(q)
    if NFKD_one == NFKC_one:
        print("same")
    else:
        print(NFKC_one,NFKD_one)


same
same
same
same
['let', 'go', 'new', 'caf', 'good'] ['let', 'go', 'new', 'cafe', 'good']
same
same


<p style = "background-color:orange; color:black;" >Actually, there is a difference which could affect the results, therefore the change in performance will be proved</p>

<h1 align="center">Inverted Index</h1>
<p>
In this section, the process of building the inverted index is presented. The inverted index is a data structure holding, for each term/word, the list of documents and other informations in which the term/word appears. Informations about the document are stored in the data structure called <b>Posting List</b>, which holds the following informations:
<ul>
    <li>Identifier of the document</li>
    <li>Occurences of the term in the document (frequency)</li>
</ul>
</p>
<p>
The function defined below <b><i>build_index</i></b> takes as input a dataset, with documents as tuples, and outputs five different data structures:
<ul>
    <li>Lexicon: a dictionary holding for each term the number of document in which it appears and the total occurences in all the collection</li>
    <li>Inverted Index: the data structure presented above</li>
    <li>Document Index: it holds for each document the length of the document itself</li>
    <li>Direct Index: it holds the mappings from the document to the tokens, having the frequency of the term in the document as value</li>
    <li>Statistics: it holds the collection statistics about the total number of documents, terms and tokens</li>
</ul>
All the data structures presented will be used by the search engine to compute the scoring functions that will rank the documents in order to return the best ones.
</p>


In [8]:
# build_index Function
from collections import defaultdict, Counter
from tqdm import tqdm
tqdm.pandas(desc='Preprocessing...')

def build_index(dataset):
    lexicon = defaultdict(lambda: [0, 0, 0])  # [termid, document frequency, term frequency]
    doc_index = []
    # defaultdict is used to tell the variable what to expect in the future of the code, in order for the system to allocate the correct amount of space
    # and implement the necessary configuration. This optimization reduces the time to build the index.
    direct_index = defaultdict(dict)
    inv_d, inv_f = defaultdict(list), defaultdict(list)

    # Altro
    termid = 0
    dataset_size = dataset.shape[0]
    total_toks = 0

    for row in tqdm(dataset.itertuples(index=False), desc='Indexing', total=dataset_size):
        # docno is taken as docid to avoid confusion in later coding. The difference is subtle, but docnos are associated to run files, therefore they need to be used in place of 
        # docids. Since it doesn't change from the point of view of identifying a document, docids are replaced with docnos. 
        docid = row.docno
        tokens = row.tokens

        token_tf = Counter(tokens)

        for token, tf in token_tf.items():
            if token not in lexicon:
                lexicon[token][0] = termid
                termid += 1

            token_id = lexicon.get(token)[0]
            inv_d[token_id].append(int(docid))
            inv_f[token_id].append(tf)
            direct_index[int(docid)][token_id] = tf
            lexicon[token][1] += 1
            lexicon[token][2] += tf

        doclen = len(tokens)
        doc_index.append((docid, doclen))
        total_toks += doclen

    stats = {
        'num_docs': dataset_size,
        'num_terms': len(lexicon),
        'num_tokens': total_toks,
    }

    return dict(lexicon), {'docids': dict(inv_d), 'freqs': dict(inv_f)}, doc_index, dict(direct_index), stats


<p>In order to tidy up the code and obtain a better data structure to work with instead of variable and functions, it was decided to define the class <b><i>InvertedIndex</i></b>. This class represents the Inverted Index data structure explained above, enriched with the methods to access the other four data structures in a safe and efficient way.</p>
<p>Inside the class it is possible to find the class <b><i>PostingListIterator</i></b> that holds all the posting lists of the inverted index, which are now accessible with simple methods. Differently from the <i>build_index</i> function, in the <i>PostingListIterator</i> class there is also a method called <i>score</i> that computes the BM25 score, which will be used to compute the rank of the documents with respect to the query given.</p>
<p<i>InvertedIndex</i> is designed to be initialized with the results of the <i>build_index</i> function.


In [9]:
# Inverted Index Class
import heapq
import math

class InvertedIndex:

    class PostingListIterator:
        def __init__(self, docids, freqs, doclens, N, avgdl, k, b):
            self.docids = docids # [1, 2, 20, etc...]
            self.freqs = freqs # [15, 30, 4, etc...]
            self.doclens = doclens # {0: 30; 1: 20 etc...}
            self.N = N
            self.avgdl = avgdl #Average Document Length
            self.K = k
            self.B = b

            self.pos = 0 # Represents the position of the iterator inside the posting list

            """
                                |
                               -V-
            self.docids = [1,   2,   20...]
            """
        def docid(self):
            if self.is_end_list():
                return math.inf
            return self.docids[self.pos]

        def score(self):
            if self.is_end_list():
                return math.inf

            tf_i = self.freqs[self.pos]
            dl_i = self.doclens.get(self.docid())
            df_i = self.len()
                                                    # ---------------- BM25 ----------------- #
            return 0 if df_i == 0 else (tf_i/(self.K * ((1 - self.B) + self.B * dl_i/self.avgdl) + tf_i)) * math.log(self.N / df_i)

        def next(self, target = None):
            if not target:
                if not self.is_end_list():
                    self.pos += 1
            else:
                if target > self.docid():
                    try:
                        self.pos = self.docids.index(target, self.pos)
                    except ValueError:
                        self.pos = len(self.docids)

        def is_end_list(self):
            return self.pos == len(self.docids)

        def len(self):
            return len(self.docids)

    def __init__(self, lex, inv, doc, direct, stats, k, b):
        self.lexicon = lex
        self.inv = inv
        self.doc = doc
        self.direct = direct
        self.stats = stats

        self.N = self.num_docs()
        self.doclens = {int(doc_entry[0]): doc_entry[1] for doc_entry in self.doc}
        total_dl = sum(doc_entry[1] for doc_entry in self.doc)
        self.avgdl = total_dl / self.N

        self.K = k # parameter of the parametric curve (saturating function) that approximates the RSV
                   # of elite terms
        self.B = b # from full (b=1) to no(b=0, TFIDF) document length normalisation

    def num_docs(self):
        return self.stats['num_docs']

    def get_posting(self, termid): # it returns the posting list for the specified term
        docids = self.inv['docids'].get(termid)
        freqs = self.inv['freqs'].get(termid)

        return InvertedIndex.PostingListIterator(
            docids=docids,
            freqs=freqs,
            doclens=self.doclens,
            N=self.N,
            avgdl=self.avgdl,
            k=self.K,
            b=self.B
        )

    def get_termids(self, tokens):
        return [self.lexicon[token][0] for token in tokens if token in self.lexicon]

    def get_postings(self, termids):
        return [self.get_posting(termid) for termid in termids]

<p>The following class <b><i>TopQueue</i></b> is used to hold the k best documents by the score obtained in the query, during the process of checking every document, in order to deliver at the end of the computation the top k documents in decreasing order of score. It is worth noticing that the class only stores k documents, therefore if there are more than k documents at the moment of insertion, the one with the worst score is removed.</p>

In [10]:
# TopQueue Class
class TopQueue:
    def __init__(self, k=10, threshold=0.0):
        self.queue = []
        self.k = k
        self.threshold = threshold

    def size(self):
        return len(self.queue)

    def would_enter(self, score):
        return score > self.threshold

    def clear(self, new_threshold=None):
        self.queue = []
        if new_threshold:
            self.threshold = new_threshold

    def __repr__(self):
        return f'<{self.size()} items, th={self.threshold} {self.queue}'

    def insert(self, docid, score):
        if score > self.threshold:
            if self.size() >= self.k:
                heapq.heapreplace(self.queue, (score, docid))
            else:
                heapq.heappush(self.queue, (score, docid))
            if self.size() >= self.k:
                self.threshold = max(self.threshold, self.queue[0][0])
            return True
        return False

<p>
In the following block it is possible to see the implementation of the <b>Document-at-a-Time</b> methodology to compute the score of documents. </p> <p> This methodology revolves around computing the score of the document (in this case, BM25) by making the sum of the score over all the terms of the query present in the document. For exploiting this solution, the posting lists must be ordered in ascending order of document ids, so that by accessing all the terms of the query at the same time it is possible to find the score of each term in the document in one go. The process starts with the lowest document's id, if a term doesn't have as first posting's document id the lowest, it means that this query term doesn't appear in that document and so goes for all the successive steps.</p><p> The power of this methodology relies on the possibility to throw away a document from the rank of documents (if its score is less than the score of the last document of the rank) as soon as it appears in the rank, while in the Term-at-a-Time methodology, the certainty about the score of a document only arrives when all the terms have been explored.
</p></p>
The functions to implement the <b>Document-at-a-Time</b> methodology of computing a rank are reported:
<ul>    
    <li><b><i>min_docid</b></i>: returns the minimum document id of the posting list given as input</li>
    <li><b><i>daat</b></i>: takes in inpututilizes the <i>min_docid</i> function to create a posting list for the term ordered in ascending order of document id</li>
    <li><b><i>query_process</b></i>: takes as input the query, the inverted index of the collection and k the number of documents to return in order to return the k documents with the highest score, obtained using the Document-at-a-Time methodology. It utilizes the function <i>daat</i> and indirectly also <i>min_docid</i> to process a query </li>
</ul>
</p>

In [11]:
def min_docid(postings):
    min_docid = math.inf
    for p in postings:
        if not p.is_end_list():
            min_docid = min(p.docid(), min_docid)
    return min_docid

def daat(postings, k=10):
    top = TopQueue(k)
    current_docid = min_docid(postings)
    
    while current_docid != math.inf:
        score = 0
        next_docid = math.inf
        for posting in postings:
            if posting.docid() == current_docid:
                score += posting.score()
                posting.next()
            if not posting.is_end_list():
                next_docid = min(posting.docid(), next_docid)
        top.insert(current_docid, score)
        current_docid = next_docid
    return sorted(top.queue, reverse=True)

def query_process(query, index, heap_size=10):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    return daat(postings, heap_size)

<p>In the next box it is possible to find the implementation of the query processing obtained using <b><i>Rocchio's Algorithm for Pseudo Relevance Feedback</b></i>.</p>
<p>In the process of retrieving documents, it is unlikely that the first k results perfectly matches the query of a user in all possible cases, mostly because common users are not capable of formulating an exhausting query. In order to obtain better results, the system needs a feedback on the retrieved documents, but it is not always possible to obtain an explicit user feedback (mark the retrieved documents to refine the search) and when even implicit user feedback is not a viable solution, there is only one option: don't use the user feedback. This last solution revolves around considering each of the k returned documents as a relevant document, so that the terms appearing in the query are included in the following query.</p><p> However documents contains usually thousands of different terms; a query with thousands of terms would result impractical to compute and most importantly, it would not retrieve the best documents. Also, there is the possibility to lose the information given by the original query.</p>
<p>In order to address this problem, an implementation of Rocchio's Algorithm has been done. Originally, the algorithm was designed for <b>explicit</b> relevance feedback, that is, knowing what documents the user classified as relevant or non relevant. This is the formula: The new query q' is composed by:
    <ul>
        <li>the original query terms, each multiplied by the factor <b>alpha</b></li>
        <li>the terms in the relevant documents multiplied by a factor <b>beta</b> and divided by the number of relevant documents</li>
        <li>the terms in the non-relevant documents multiplied by: -1;a factor <b>gamma</b> and divided by the number of non relevant documents</li>
    </ul>
    Since there is not the information about non-relevant documents, in the Pseudo-Relevance implementation of Rocchio's Algorithm, gamma is put to zero. In this implementation, a value of 8 for alpha and 16 for beta was chosen, along with the decision to take only the 10 terms with the highest weight, among all the possible terms.
</p>

In [12]:
def daat_with_weights(postings, k=10, weighted = False):
    top = TopQueue(k)
    posting_lists = postings.keys()
    current_docid = min_docid(posting_lists)
    
    while current_docid != math.inf:
        score = 0
        next_docid = math.inf
        for posting, weight in postings.items():
            if posting.docid() == current_docid:
                score += posting.score() * weight # moltiplico per il peso nel caso di nuovo vettore
                posting.next()
            if not posting.is_end_list():
                next_docid = min(posting.docid(), next_docid)
        top.insert(current_docid, score)
        current_docid = next_docid
    return sorted(top.queue, reverse=True)


# Rocchio's parameters
alpha = 8
beta = 16
def rocchio_pseudo_feedback(query_terms, doc_ids, lexicon, direct_index):
    # the following code executes the Rocchio's algorithm in the context of Pseudo Relevance, therefore there
    # no non-relevant documents (gamma is put to 0). More on that, in the successive call of the function "query_process_with_rocchio"
    # only the first ten query terms by weight are used in the search. The function takes as input the query terms, the document ids
    # the lexicon and the direct index, the latter will be used to find the new query terms to compute the new query vector. The function
    # returns as output the new query vector composed of the entire set of terms of the relevant documents. the selection of the top ten query terms will be done later,
    # as it has already been said.

    query_vector = defaultdict(int) # creation of the vector of the query
    for term in set(query_terms): # the usage of set optimizes the search
        if term in lexicon: # unknown terms will receive 0 as value
            query_vector[lexicon[term][0]] = 1 # the dict is indicized by term id
##########
    relevant_vector = defaultdict(float) # creation of the vector of relevant documents
    doc_set = set(doc_ids) # switched to using set for efficiency reason like for what was done for the query terms
            
    for doc in doc_set: # for every document id in the set of document ids
        for term_id, tf in direct_index[doc].items(): # take the term_id and the frequency of the term from the indicized document in the direct index
            relevant_vector[term_id] += tf # add to the frequency of the term the frequency found in the direct index
    # by doing so, the frequencies of the terms present in the RELEVANT Documents returned by TopQueue are compacted into one dictionary: there is an entry for each term, which 
    # contains as value the sum of the frequencies of the term in all the documents returned. In this way, computing the weights results to be easier, since the overal frequency is
    # already computed. 
    
    new_query_vector = defaultdict(float)
    for term_id in query_vector.keys() | relevant_vector.keys(): # the iteration is over all terms: initial query terms and terms extracted from relevant documents
        # term id is a number, therefore there is the need to utilize inv_lexicon that maps the term_id to the term in string
        new_query_vector[inv_lexicon[term_id]] = (alpha * query_vector[term_id]) + (beta * relevant_vector[term_id] / len(doc_ids))

    return new_query_vector


def query_process_with_rocchio(query, index, heap_size=10, max_rel_docs=5):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    res = daat(postings, max_rel_docs)
    
    relevant_docids = [r[1] for r in res]
    
    new_query_terms = rocchio_pseudo_feedback(qtokens, relevant_docids, index.lexicon, index.direct)
    qtermids = index.get_termids(new_query_terms.keys())
    postings = index.get_postings(qtermids)
    
    weighted_postings = {}
    for term_weight, posting in zip(new_query_terms.values(), postings):
        weighted_postings[posting] = term_weight
    top_terms = dict(sorted(weighted_postings.items(), key=lambda item: item[1], reverse=True)[:10]) # solo i primi 10 termini
    
    return daat_with_weights(top_terms, heap_size)

<p>At this point, the search engine is basically complete and functioning: the preprocessing functions transform the query into single terms that are utilized to compute the score of the documents either using BM25 only or BM25 and Rocchio's algorithm. Now what is left is evaluate the search engine. It is certain that it retrieves documents, but are they really relevant to the query given ?</p>
<p>Like for a machine learning classificator's evalutaion, in order to evaluate the performance of a search engine, there must be some data that can be used to test the performance; in particular for this task there is the need to have a lot of queries each of which must be associated with some relevant documents that must be present in the collection of documents.</p>
<p>The evaluation of a search engine, in details, makes use of big collections of documents composed by:
    <ul>
        <li><b>The documents</b>: used to build the Inverted Index and all the structures needed</li>
        <li><b>Topics</b>: surrogates for information needs of a user. They are made of three parts: <b>title</b>, <b>description</b>, <b>narrative</b>. the first component (title) is what search engine will use as a query to retrieve relevant documents. The other two components are used in the relevance assessment phase to decid if a document is relevant or not for the query (the title).</li>
        <li><b>Relevance judgements (relevance assessments or qrels)</b>: values that represent the relevance of the document with respect to the query. They can be binary (relevant, non relevant) or graded (0=not relevant, 5=very relevant)</li>
    </ul>
    The evaluation process continues by giving the topics to the search engine an making it retrieving the documents. The result is a <b>run file</b>, a textual file containing, for each of the tipically 50000 tuples: 
    <ul>
        <li><b>Topic ID</b>: it is the identifier of the topic in the test collection used.</li>
        <li><b>Fixed</b>: a field containing always the same value. originally inserted for specific purposes, nowadays unused and kept only for legacy reasons.</li>
        <li><b>Document No</b>: it is an identifier of the document. Normally it is different from the Document ID, but in this project, as mentioned before, Document IDs contain the Document Number, in this way the computation is easier. The difference between Document IDs and Document No is that Document IDs are identifiers that can be created at run time that are used to identify documents, but they can differ from implementation to implementation, since even integer numbers can be used as identifiers. Document Nos, instead, are associated before hand, in this way the same Document No is always associated to the same document in all implementations. Without this convention, it would be impossible to keep track of documents and comparing would be impossible.</li>
        <li><b>Rank</b>: the position in the documents retrieval for a specific query that a specific document got, based on its score.</li>
        <li><b>Score</b>: the result of the scoring function used that a document got over a specific query (title of the topic)</li>
        <li><b>Run ID</b>: the identifier of the run, in order to not confuse two different run files in case they are merged together</li>
    </ul>
    Tipically, in a run file there are 50 topics with 1000 retrieved documents for each topic. Once the run file is obtained, <b>Topic ID</b> and <b>Document No</b> are used to retrieve from the qrels the relevance of that document for that query (topic), and then the final result is used with some metrics to obtain the results about the performance of the search engine. 
</p>
<p>
In the following box there are three functions, used to implement the creation of a run file and the qrels file: The first function <b>create_run_file</b> creates the run file over the topics contained in the section of MSMARCO's dataset "test-2020" for both the search engine of this project (if the inverted index is given) and the search engine of MSMARCO used as baseline (if no inverted index is given). The search engine used by MSMARCO will be presented in the following sections; <b>create_run_file_with_rocchio</b> shares a similar role with the previous function, but in this case, it provides only the run file for the search engine of this project, obtained utilizing Rocchio's Algorithm for pseudo relevance feedback over the same topics and documents.
The last function <b>create_qrels_file</b> instead is used to create the file that maps Document No and Topic ID to the relevance of the document for that query. The file contains 4 columns: Topic id; fixed value; Document number; relevance value.
</p>

In [13]:
def create_run_file(dataset, inv_index, name):
    topics = dataset.get_topics("test-2020")
    k = 1000
    results = []

    if inv_index:
        # this side of the condition consent the computation of the run file 
        # for the search engine of this project
        for query in tqdm(topics.itertuples(index=False), desc='Creating run file', total=len(topics)):
            res, query_id = query_process(query.query, inv_index, k), query.qid
            results.extend([f"{query_id} Q0 {r[1]} {rank} {r[0]} run.txt\n" for rank, r in enumerate(res, start=1)])
    else:
        # this side of the condition consent the retrieval of the run file 
        # of the search engine of MSMARCO, whose implementation will be presented
        # in the following boxes
        for result in tqdm(topics.itertuples(index=False), desc='Creating run file', total=len(topics)):
            query_id, docno, rank, score = result.qid, result.docno, result.rank, result.score
            results.append(f"{query_id} Q0 {docno} {rank} {score} run.txt\n")

    with open(name, "w") as f:
        f.writelines(results)

def create_run_file_with_rocchio(dataset, inv_index, name, max_rel_docs=5):
    dataset = dataset.get_topics("test-2020")
    k = 1000
    results = []

    for query in tqdm(dataset.itertuples(index=False), desc='Creating run file', total=len(dataset)):
        res, query_id = query_process_with_rocchio(query.query, inv_index, k, max_rel_docs), query.qid
        results.extend([f"{query_id} Q0 {r[1]} {rank} {r[0]} run.txt\n" for rank, r in enumerate(res, start=1)])

    with open(name, "w") as f:
        f.writelines(results)

def create_qrels_file(dataset, name):
    qrels_df = dataset.get_qrels("test-2020")
    qrels = []

    for qrel in tqdm(qrels_df.itertuples(index=False), desc='Creating qrels file', total=len(qrels_df)):
        query_id, docno, label = qrel.qid, qrel.docno, qrel.label
        qrels.append(f"{query_id} 0 {docno} {label}\n")

    with open(name, "w") as f:
        f.writelines(qrels)

<p><b>trec_test</b> function is utilized to compute the values of different metrics, given as input: the dataset (MSMARCO); the inverted index of the search engine; the name to give to the run file; boolean value to decide to use Rocchio's algorithm or not; maximum number of documents to retrieve in the pseudo relevance feedback. It returns as output the lists of values for each metric used.</p>
<p>The metrics to assess the performance of the search engine are:
<ul>
    <li><b>Precision at cutoff k (P@k)</b>: it is the sum of the relevance of the first k documents retrieved over k. For this metric, in the project k is equal to 5</li>
    <li><b>Precision at cutoff k with relevance m (P(rel=m)@k)</b>: it is the same formula as the precision, the difference relies in what documents are considered relevant: in  P@k, every document with a grade strictly greater than 0 is considered relevant; in this case instead the documents considered relevant are the ones having a grade greater or equal than m. For this metric in the project m is equal to 2 and k is equal to 5</li>
    <li><b>Normalized Discounted Cumulative Gain at cutoff k (nDCG@k)</b>: given the parameter b that represents the patience of the user (2=impatient, 10=patient) the Discounted Cumulative Gain at cutoff k (DCG@k) is:
    <ul>
        <li>Sum of the relevance of the retrieved documents if k is strictly less than b</li>
        <li>DCG@k-1 summed with the ratio between the relevance of the k<sup>th</sup> document and log<sub>b</sub>(k)</li>
    </ul>
    therefore, this is a recursive formula. The Normalized Discounted Gain at cutoff k, used in this project, is the ratio between DCG@k and the ideal run, that is, the sum of the relevance of the k most relevant documents, ordered in descending order of relevance. The decision to use nDCG@k in favor of DCG@k relies on the fact that the first metric is bounded between 0 and 1, while the second has the maximum unbounded, making it more difficult to interpret. For this metric in the project, k is equal to 10.
    </li>
    <li><b>Average Precision (AP)</b>: it is the arithmetic mean of P@k for all the possible values of k from 1 to the total number of relevant documents for the query.</li>
    <li><b>Average Precision with relevance m (AP(rel=m))</b>: it is the same formula as for Average Precision but instead of considering relevant all the documents with a grade greater or equal than 1, in this metric a document must have a grade greater or equal than m to be considered relevant. For this metric in the project m is equal to 2.</li>
</ul>

</p>

In [14]:
import ir_measures
def trec_test(dataset, inv_index=None, text_file="run_file.txt", rocchio=False, pseudo_rel_max_doc=0):
    if rocchio:
        create_run_file_with_rocchio((dataset, inv_index, pseudo_rel_max_doc, text_file))
    else:
        create_run_file(dataset, inv_index, text_file)

    qrels = list(ir_measures.read_trec_qrels('qrels.txt'))
    run = list(ir_measures.read_trec_run(text_file))
    
    print(ir_measures.calc_aggregate([P@5, P(rel=2)@5, nDCG@10, AP, AP(rel=2)], qrels, run))

    Pat5_list = {m.query_id: m.value for m in ir_measures.iter_calc([P@5], qrels, run)}
    PRel2at5_list = {m.query_id: m.value for m in ir_measures.iter_calc([P(rel=2)@5], qrels, run)}
    nDCGat10_list = {m.query_id: m.value for m in ir_measures.iter_calc([nDCG@10], qrels, run)}
    AP_list = {m.query_id: m.value for m in ir_measures.iter_calc([AP], qrels, run)}
    APRel2_list = {m.query_id: m.value for m in ir_measures.iter_calc([AP(rel=2)], qrels, run)}
    
    return Pat5_list, PRel2at5_list, nDCGat10_list, AP_list, APRel2_list

<p>The following function <i>boxplot_and_results</i> is used create a boxplot of the metrics in order to visualize the performances.</p>

In [15]:
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

def boxplot_and_results(data, metrics):
    for dicts, metric in zip(data, metrics):
        our_dict, baseline_dict = dicts[0], dicts[1]
        our_values = list(our_dict.values())
        baseline_values = list(baseline_dict.values())
    
        stat, p = stats.ttest_ind(our_values, baseline_values)
        print(f"T - test: statistic = {stat}, p = {p}")
    
        plt.figure(figsize=(8, 3))
        plt.title(metric)
        bplot = plt.boxplot([our_values, baseline_values],
                            vert=True,
                            patch_artist=True,
                            tick_labels=['Our', 'Baseline'])
        plt.show()

<p align="center" style="height:100px; background-color:red;">TEST DA VEDERE COME METTERE !!!</p>

In [16]:
# TEST PER IL PREPROCESSING
queries = [
    "M4rk is a good programmer from U.S.A.!",
    "www.kaggle.com is a cool website, i love it!",
    "click this link for free 1000000 v-bucks, http://free-vbucks.com",
    "This is my new email, new_email@google.com",
    "Let`s go to this new Cafè, it's so good ♫ ☺!",
    "SPAM SPAM SPAM COPY COPY SPAM SPAM",
    "thissentenceisattachedbutwithwordninjawecansplitit"
]
for q in queries:
    print(preprocess(q))

['m4rk', 'good', 'programm', 'usa']
['cool', 'websit', 'love']
['click', 'link', 'free', '1000000', 'v', 'buck']
['new', 'email']
['let', 'go', 'new', 'cafe', 'good']
['spam', 'copi', 'spam']
['sentenc', 'attach', 'word', 'ninja', 'split']


In [17]:
import pandas as pd

# TEST TO SEE IF THE BM25 FORMULA IMPLEMENTED IS CORRECT
test_docs = [['The cat playing on the bed', '1'], ['Nick is playing football', '2'], ['Mark is whatching football on the bed', '3']]
test_df = pd.DataFrame(test_docs, columns=['text', 'docno'])
print(test_df)
print()

test_df["tokens"] = test_df["text"].apply(preprocess)
print(test_df)

t_lex, t_inv, t_doc, t_direct, t_stats = build_index(test_df)
print(t_lex)
print(t_inv)
print(t_doc)
print(t_direct)
print(t_stats)

                                    text docno
0             The cat playing on the bed     1
1               Nick is playing football     2
2  Mark is whatching football on the bed     3

                                    text docno                        tokens
0             The cat playing on the bed     1              [cat, play, bed]
1               Nick is playing football     2         [nick, play, footbal]
2  Mark is whatching football on the bed     3  [mark, whatch, footbal, bed]


Indexing: 100%|██████████| 3/3 [00:00<00:00, 23696.63it/s]

{'cat': [0, 1, 1], 'play': [1, 2, 2], 'bed': [2, 2, 2], 'nick': [3, 1, 1], 'footbal': [4, 2, 2], 'mark': [5, 1, 1], 'whatch': [6, 1, 1]}
{'docids': {0: [1], 1: [1, 2], 2: [1, 3], 3: [2], 4: [2, 3], 5: [3], 6: [3]}, 'freqs': {0: [1], 1: [1, 1], 2: [1, 1], 3: [1], 4: [1, 1], 5: [1], 6: [1]}}
[('1', 3), ('2', 3), ('3', 4)]
{1: {0: 1, 1: 1, 2: 1}, 2: {3: 1, 1: 1, 4: 1}, 3: {5: 1, 6: 1, 4: 1, 2: 1}}
{'num_docs': 3, 'num_terms': 7, 'num_tokens': 10}


In [18]:
query = "Nick football"
print(preprocess(query))
# total doc length = 10
# avg doc length = 3.333
# k = 1.2 as eg.
# b = 0.7 as eg.
# math.log è in base e
# doc 1 -> BM25 = 0
# doc 2 -> BM25 = [ Nick, (1 * log(3/1))/(1.2*((1-0.7) + 0.7*(3/3.33)) + 1) = 0.519; 
#                  football, (1 * log(3/2))/(1.2*((1-0.7) + 0.7*(3/3.33)) + 1) = 0.1915] = 0.7105
# doc 3 -> BM25 = [ football, (1 * log(3/2))/(1.2*((1-0.7) + 0.7*(4/3.33)) + 1) ] = 0.171
test_inv_index = InvertedIndex(t_lex, t_inv, t_doc, t_direct, t_stats, 1.2, 0.7)
res = query_process(query, test_inv_index, 10)
res # i risultati combaciano

['nick', 'footbal']


[(0.7108116241853848, 2), (0.1712268193024343, 3)]

mostrare con la PCA, plottare i vettori prima e dopo l'algoritmo di rocchio

Plottare in un grafico le lunghezze dei documenti dopo aver calcolato il direct index

<p align="center" style="height:100px; background-color:red;">Fine dei test</p>

<h1 align='center'>IMPLEMENTATION</h1>
<p>In this section, the data structures and functions presented before are going to be used to implement a search engine on the collection of document <b>MSMarco</b>. </p>
<p>The first step consists in reading the tsv file holding the collection of documents using the package <i>polars</i> which is an alternative of the package <i>pandas</i> fine-tuned to read large files in small amounts of time</p>

In [19]:
import polars as pl

tsv_file = 'collection.tsv'

temp = pl.read_csv(tsv_file, separator='\t', has_header=False, new_columns=['docno', 'text'], encoding="utf-8").to_pandas()

In [20]:
temp["tokens"] = temp["text"].progress_apply(preprocess)

stem_word.cache_clear()
split_word.cache_clear()

Preprocessing...: 100%|██████████| 8841823/8841823 [05:21<00:00, 27500.03it/s]


In [21]:
lex, inv, doc, direct, stats = build_index(temp)
print(stats)

Indexing: 100%|██████████| 8841823/8841823 [03:31<00:00, 41827.62it/s]

{'num_docs': 8841823, 'num_terms': 1205151, 'num_tokens': 307062371}


In [22]:
import joblib

inv_lexicon = {v[0]: k for k, v in lex.items()} # Per ottenere come risultato una query di stringhe e non numeri

with open("files\\lex.pkl", "wb") as f:
    joblib.dump(lex, f)
    del lex
gc.collect()




30

In [ ]:
with open("files\\inv.pkl", "wb") as f:
    joblib.dump(inv, f)
    del inv
with open("files\\doc.pkl", "wb") as f:
    joblib.dump(doc, f)
    del doc
with open("files\\direct.pkl", "wb") as f:
    joblib.dump(direct, f)
    del direct
with open("files\\stats.pkl", "wb") as f:
    joblib.dump(stats, f)
    del stats
gc.collect()

In [ ]:
lex, inv, doc, direct, stats = joblib.load("files\\lex.pkl"),\
                    joblib.load("files\\inv.pkl"),\
                    joblib.load("files\\doc.pkl"),\
                    joblib.load("files\\direct.pkl"),\
                    joblib.load("files\\stats.pkl"),

In [ ]:
inv_index = InvertedIndex(lex, inv, doc, direct, stats, 1.2, 0.7)

with open("files\\inv_index.pkl", "wb") as f:
    joblib.dump(inv_index, f)

del lex, inv, doc, direct, stats
gc.collect()

In [ ]:
dataset = pt.datasets.get_dataset("msmarco_passage") # per le query
create_qrels_file(dataset, "qrels.txt")

In [ ]:
# PYTERRIER TEST #
bm25_terrier_stemmed = pt.terrier.Retriever.from_dataset('msmarco_passage', 'terrier_stemmed', wmodel='BM25')

pipeline = bm25_terrier_stemmed
pipeline(dataset.get_topics("test-2020"))

In [ ]:
OurPat5, OurPRel2at5, OurnDCGat10, OurAP, OurAPRel2 = trec_test(dataset, inv_index, "run.txt")
BaselinePat5, BaselinePRel2at5, BaselinenDCGat10, BaselineAP, BaselineAPRel2 = trec_test(dataset, None, "ptrun.txt")

<p>SPIEGAZIONE DELLE METRICHE</p>

In [ ]:
data = [(OurPat5, BaselinePat5),
        (OurPRel2at5, BaselinePRel2at5),
        (OurnDCGat10, BaselinenDCGat10),
        (OurAP, BaselineAP),
        (OurAPRel2, BaselineAPRel2)]
metrics = ["P@5", "P(rel=2)@5", "nDCG@10", "AP", "AP(rel=2)"]

boxplot_and_results(data, metrics)

In [ ]:
RocchioPat5, RocchioPRel2at5, RocchionDCGat10, RocchioAP, RocchioAPRel2 = trec_test(dataset, inv_index, "run.txt", True, 3)

new_data = [(RocchioPat5, BaselinePat5),
            (RocchioPRel2at5, BaselinePRel2at5),
            (RocchionDCGat10, BaselinenDCGat10),
            (RocchioOurAP, BaselineAP),
            (RocchioOurAPRel2, BaselineAPRel2)]

boxplot_and_results(new_data, metrics)

In [ ]:
def profile(f):
    def f_timer(*args, **kwargs):
        start = time.time()
        result = f(*args, **kwargs)
        end = time.time()
        ms = (end - start) * 1000
        print(f"{f.__name__} ({ms:.3f} ms)")
        return result
    return f_timer

In [ ]:
@profile
def timed_query_process(query, index, heap_size=10):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    return daat(postings, heap_size)

In [ ]:
@profile
def timed_query_process_with_rocchio(query, index, heap_size=10, max_rel_docs=5):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    res = daat(postings, max_rel_docs)
    
    relevant_docids = [r[1] for r in res]
    
    new_query_terms = rocchio_pseudo_feedback(qtokens, relevant_docids, index.lexicon, index.direct)
    qtermids = index.get_termids(new_query_terms.keys())
    postings = index.get_postings(qtermids)
    
    weighted_postings = {}
    for term_weight, posting in zip(new_query_terms.values(), postings):
        weighted_postings[posting] = term_weight
    top_terms = dict(sorted(weighted_postings.items(), key=lambda item: item[1], reverse=True)[:10])
    
    return daat_with_weights(top_terms, heap_size)

In [ ]:
query_results = dict()

for query in dataset.get_topics("test-2020")[:5].itertuples(index=False):
    res = timed_query_process(query.query, inv_index)
    query_results[query.query] = []
    for r in res:
        row = temp.loc[temp["docno"] == r[1]]
        query_results[query.query].append((row.docno.values, row.text.values))

for q, res in query_results.items():
    print(q)
    print()
    for r in res:
        print(r[1])
    print("----------")

In [ ]:
query_results = dict()

for query in dataset.get_topics("test-2020")[:5].itertuples(index=False):
    res = timed_query_process_with_rocchio(query.query, inv_index, max_rel_docs=3)
    query_results[query.query] = []
    for r in res:
        row = temp.loc[temp["docno"] == r[1]]
        query_results[query.query].append((row.docno.values, row.text.values))

for q, res in query_results.items():
    print(q)
    print()
    for r in res:
        print(r[1])
    print("----------")